# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`

This notebook demonstrates how to explore and process the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

We will guide you through loading metadata, inspecting available record sets, extracting data, processing it, and visualizing key features. Throughout, **all entities** (record sets, fields, columns) are referenced by their `@id` as required by Croissant best practices.

### Dataset Source
Croissant schema URL: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Instantiate mlcroissant Dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Let's examine what record sets and fields are available in this package. All entities are referenced by their `@id` field.

We will list all available record sets by their `@id`, then show all fields for each record set with the associated types.

In [ ]:
# List all record sets by @id
print('Available Record Sets:')
record_sets = []
for record_set in metadata.record_sets:
    print(f"- @id: {record_set.id}, name: {record_set.name}")
    record_sets.append(record_set.id)

# List all fields in each record set, referenced by their @id
import pprint
for record_set in metadata.record_sets:
    print(f"\nRecord Set: {record_set.id} ({record_set.name})")
    print("Fields:")
    for field in record_set.fields:
        print(f"  - @id: {field.id}, name: {field.name}, type: {field.data_type}")

## 3. Data Extraction
Load data from the main record set into a pandas DataFrame for analysis.

We use each record set's `@id` to extract records. Below, we'll focus on the main clinical data table. If more than one record set exists, you can extend as needed.

In [ ]:
# For this dataset, there should be only one main record set with the clinical data
# Use its @id to load records. We'll load all record sets for demonstration.
dfs = {}
for rs_id in record_sets:
    print(f"\nLoading data for record set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if len(records) == 0:
        print("  (No records in this record set.)")
        continue
    df = pd.DataFrame(records)
    dfs[rs_id] = df
    print(f"  Loaded {len(df)} records with columns:\n    {list(df.columns)}")

# Choose the main data table: If only one record set, select it; otherwise, ask user.
main_record_set_id = record_sets[0] if record_sets else None

if main_record_set_id is not None and main_record_set_id in dfs:
    main_df = dfs[main_record_set_id]
    print("\nColumns in main record set:")
    print(main_df.columns.tolist())
    print(main_df.head())
else:
    print("No main record set data available.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering, normalization, and grouping, referencing field `@id`s explicitly as per Croissant guidelines.

Below, we demonstrate:
- Filtering the dataset for patients above a certain age
- Normalizing the age field
- Grouping by sex (if available)

_Replace field `@id`s with the actual ones listed above when experimenting further._

In [ ]:
# Map of possible field name or @id for age and sex
from difflib import get_close_matches

df = main_df.copy()

# Find the age-related field: search for fields containing 'age' in their @id or column name
age_field_candidates = [col for col in df.columns if 'age' in col.lower()]
if age_field_candidates:
    age_field_id = age_field_candidates[0]
else:
    age_field_id = df.columns[0]  # fallback

# Find the sex/gender field
sex_field_candidates = [col for col in df.columns if any(x in col.lower() for x in ['sex','gender'])]
if sex_field_candidates:
    sex_field_id = sex_field_candidates[0]
else:
    sex_field_id = None

print(f"Using @id for age field: {age_field_id}")
if sex_field_id:
    print(f"Using @id for sex field: {sex_field_id}")

# Coerce age to numeric if not already
df[age_field_id] = pd.to_numeric(df[age_field_id], errors='coerce')

# Remove outliers: keep only realistic ages (e.g., 18 < age < 100)
df = df[df[age_field_id].between(18, 100)]

# Filter for age > 60
threshold = 60
filtered_df = df[df[age_field_id] > threshold].copy()
print(f"Filtered records with {age_field_id} > {threshold} (n={len(filtered_df)}):")
print(filtered_df.head())

# Normalize the age field
norm_col = f"{age_field_id}_normalized"
filtered_df[norm_col] = (filtered_df[age_field_id] - filtered_df[age_field_id].mean()) / filtered_df[age_field_id].std()
print(f"\nNormalized {age_field_id} for filtered records:")
print(filtered_df[[age_field_id, norm_col]].head())

# Group by sex if available, and compute average normalized age
if sex_field_id and sex_field_id in filtered_df.columns:
    grouped = filtered_df.groupby(sex_field_id)[[age_field_id, norm_col]].mean()
    print(f"\nGrouped by {sex_field_id} (mean age and normalized age):")
    print(grouped.head())

## 5. Visualization
Visualize the age distribution and sex-specific summary, using the field `@id`s discovered above.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,5))
sns.histplot(df[age_field_id].dropna(), bins=12, kde=True)
plt.xlabel(f"Age ({age_field_id})")
plt.title(f"Distribution of Age ({age_field_id})")
plt.show()

if sex_field_id and sex_field_id in df.columns:
    plt.figure(figsize=(8,5))
    sns.boxplot(data=df, x=sex_field_id, y=age_field_id)
    plt.xlabel(sex_field_id)
    plt.ylabel(age_field_id)
    plt.title(f"Age by Sex Grouping ({sex_field_id})")
    plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to use the `mlcroissant` library to:
- Access dataset metadata and understand the available structure via `@id` fields
- Extract and analyze tabular records using proper Croissant practices
- Apply common preprocessing and visual EDA steps to explore demographic patterns

You can extend this notebook by referencing further `@id`s of relevant fields and performing additional clinical outcome or biomarker analyses.

**Remember:** When working with sensitive patient data, always follow privacy and ethical guidelines in all downstream applications.